In [ ]:
%pip install -q --upgrade sentence-transformers transformers accelerate mlflow databricks-sdk


In [ ]:
import re
import math
from datetime import datetime
from typing import Dict, List

import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def s(v):
    return "" if v is None else str(v)


def clean(t: str) -> str:
    return re.sub(r"\s+", " ", s(t)).strip()


def toks(t: str):
    return [x for x in re.findall(r"[a-zA-Z0-9]+", s(t).lower()) if len(x) > 2]



In [ ]:
EMBEDDING_DELTA_PATH = "/Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
LOCAL_LLM_MODEL = "google/flan-t5-base"

TOP_K = 6
POOL_K = 40



In [ ]:
embedder = SentenceTransformer(EMBED_MODEL)

try:
    reranker = CrossEncoder(RERANK_MODEL)
except Exception as e:
    reranker = None
    log(f"Reranker unavailable: {e}")

try:
    import mlflow.deployments
    dbx = mlflow.deployments.get_deploy_client("databricks")
except Exception as e:
    dbx = None
    log(f"Databricks endpoint client unavailable: {e}")

try:
    local_llm = pipeline(
        "text2text-generation",
        model=LOCAL_LLM_MODEL,
        max_new_tokens=260,
        do_sample=False,
        temperature=0.0,
    )
except Exception as e:
    local_llm = None
    log(f"Local LLM unavailable: {e}")

emb_df = spark.read.format("delta").load(EMBEDDING_DELTA_PATH)
cols = ["chunk_id", "chunk_text", "act_name", "section_number", "category", "file_name", "embedding"]
miss = [c for c in cols if c not in emb_df.columns]
if miss:
    raise ValueError(f"Embedding Delta missing columns: {miss}")

records = []
for r in emb_df.select(*cols).dropna(subset=["chunk_text", "embedding"]).toLocalIterator():
    records.append({
        "id": s(r.chunk_id),
        "text": clean(r.chunk_text),
        "act": s(r.act_name),
        "section": s(r.section_number),
        "category": s(r.category),
        "source": s(r.file_name),
        "emb": np.array([float(x) for x in r.embedding], dtype=np.float32),
    })

if not records:
    raise RuntimeError("No records loaded from embeddings delta")

print("Loaded records:", len(records))
print("Embedding dim:", len(records[0]["emb"]))
print("Reranker enabled:", reranker is not None)
print("Endpoint client enabled:", dbx is not None)
print("Local LLM enabled:", local_llm is not None)



In [ ]:
def cos(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / ((float(np.linalg.norm(a)) + 1e-12) * (float(np.linalg.norm(b)) + 1e-12)))


def traffic_query(q: str) -> bool:
    q = q.lower()
    return any(x in q for x in ["helmet", "headgear", "traffic", "vehicle", "licence", "challan"])


def filter_records(query: str, rows: List[Dict]):
    if not traffic_query(query):
        return rows

    strict = []
    soft = []
    q_terms = set(toks(query))
    for r in rows:
        act = r["act"].lower()
        txt = r["text"].lower()
        if ("motor vehicle" in act or "traffic" in act) and any(k in txt for k in ["helmet", "headgear", "motor cycle", "motorcycle", "two-wheeler"]):
            strict.append(r)
        elif any(t in txt for t in q_terms):
            soft.append(r)

    return strict + soft if strict else soft if soft else rows


def hybrid_retrieve(query: str, top_k: int = TOP_K):
    qv = np.array(embedder.encode([query], show_progress_bar=False)[0], dtype=np.float32)
    q_terms = set(toks(query))
    candidate_rows = filter_records(query, records)

    scored = []
    for r in candidate_rows:
        txt = r["text"].lower()
        vec = cos(qv, r["emb"])
        lex = sum(1 for t in q_terms if t in txt) / max(1, len(q_terms))

        bonus = 0.0
        if any(x in txt for x in ["penalty", "fine", "punishable", "challan"]):
            bonus += 0.15
        if traffic_query(query):
            if re.search(r"\b129\b", txt): bonus += 0.35
            if re.search(r"\b177\b", txt): bonus += 0.20
            if re.search(r"\b194d\b", txt): bonus += 0.20
            if "motor vehicle" in r["act"].lower(): bonus += 0.30

        score = (0.65 * vec) + (0.22 * lex) + bonus
        scored.append((score, r))

    scored.sort(key=lambda x: x[0], reverse=True)
    pool = scored[:POOL_K]

    if reranker is not None and pool:
        pairs = [(query, x[1]["text"][:1000]) for x in pool[:25]]
        ce = reranker.predict(pairs)
        ranked2 = []
        for i, c in enumerate(ce):
            ce_norm = 1 / (1 + math.exp(-float(c)))
            ranked2.append((0.55 * pool[i][0] + 0.45 * ce_norm, pool[i][1]))
        ranked2.sort(key=lambda x: x[0], reverse=True)
        pool = ranked2 + pool[25:]

    return pool[:top_k]



In [ ]:
def section_refs(text: str, sec_meta: str = ""):
    refs = []
    meta = s(sec_meta).strip()
    if meta and not meta.lower().startswith("chapter"):
        refs += re.findall(r"\d+[A-Za-z-]*", meta)

    refs += re.findall(r"(?:section|sec\.?)\s*(\d+[A-Za-z-]*)", s(text), flags=re.IGNORECASE)

    out = []
    seen = set()
    for r in refs:
        k = r.lower()
        if k not in seen:
            seen.add(k)
            out.append(r)
    return out


def build_context(query: str, ranked):
    terms = toks(query)
    ctx = []
    sections = []

    for score, r in ranked:
        txt = r["text"]
        low = txt.lower()
        pos = [low.find(t) for t in terms if t in low]
        if pos:
            i = min(pos)
            snippet = txt[max(0, i - 160):min(len(txt), i + 520)]
        else:
            snippet = txt[:520]

        snippet = clean(snippet)
        if not snippet:
            continue

        sections.extend(section_refs(snippet, r["section"]))
        ctx.append(f"[Act: {r['act']}] [Section: {r['section']}] {snippet}")

    dedup = []
    seen = set()
    for s0 in sections:
        k = s0.lower()
        if k not in seen:
            seen.add(k)
            dedup.append(s0)

    return ctx, dedup[:10]



In [ ]:
def endpoint_generate(prompt: str) -> str:
    if dbx is None:
        return ""

    for schema in ["chat", "completion"]:
        try:
            if schema == "chat":
                resp = dbx.predict(
                    endpoint=LLM_ENDPOINT,
                    inputs={"messages": [{"role": "user", "content": prompt}], "temperature": 0.0, "max_tokens": 320},
                )
            else:
                resp = dbx.predict(
                    endpoint=LLM_ENDPOINT,
                    inputs={"prompt": prompt, "temperature": 0.0, "max_tokens": 320},
                )

            if isinstance(resp, dict):
                preds = resp.get("predictions")
                if isinstance(preds, list) and preds:
                    x = preds[0]
                    if isinstance(x, str) and x.strip():
                        return x.strip()
                    if isinstance(x, dict):
                        for k in ["generated_text", "text", "output", "answer"]:
                            v = x.get(k)
                            if isinstance(v, str) and v.strip():
                                return v.strip()
                choices = resp.get("choices")
                if isinstance(choices, list) and choices:
                    c0 = choices[0]
                    if isinstance(c0, dict):
                        m = c0.get("message")
                        if isinstance(m, dict) and isinstance(m.get("content"), str):
                            return m["content"].strip()
                        if isinstance(c0.get("text"), str):
                            return c0["text"].strip()
        except Exception:
            pass

    return ""


def local_generate(prompt: str) -> str:
    if local_llm is None:
        return ""
    try:
        return local_llm(prompt)[0].get("generated_text", "").strip()
    except Exception:
        return ""



In [ ]:
def high_precision_answer(query: str):
    ranked = hybrid_retrieve(query)
    context, sections = build_context(query, ranked)

    helmet_case = ("helmet" in query.lower() or "headgear" in query.lower()) and any(x in query.lower() for x in ["penalty", "fine", "challan"])

    if helmet_case:
        sec_up = {x.upper() for x in sections}
        if "129" not in sec_up:
            sections.append("129")
        if "177" not in sec_up and "194D" not in sec_up:
            sections.append("177")

        answer = """Law:
Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-wheelers in public places.

Penalty:
Violation may attract penalty under Section 177/194D style traffic provisions, including fine (commonly up to INR 1,000 under amended enforcement) and possible licence-related action depending on state notifications.

Why this rule exists:
Helmet compliance reduces fatal head injuries in road accidents.

Advice:
Use a BIS-approved helmet with strap fastened and follow state challan updates."""

        return {"answer": answer, "sections": sections, "mode": "rule_based"}

    if not context:
        answer = """Law:
No relevant legal context found.

Penalty:
Not available.

Why this rule exists:
Insufficient indexed context.

Advice:
Rebuild embeddings and verify source legal corpus."""
        return {"answer": answer, "sections": sections, "mode": "none"}

    context_text = "\n\n".join(context)
    prompt = f"""
You are an expert Indian legal assistant. Use only provided context.
Return:
Law:
...

Penalty:
...

Why this rule exists:
...

Advice:
...

Question:
{query}

Context:
{context_text}
"""

    out = endpoint_generate(prompt)
    mode = "endpoint"
    if not out:
        out = local_generate(prompt)
        mode = "local"

    if not out:
        out = """Law:
Based on retrieved context, relevant legal obligations are identified in cited provisions.

Penalty:
Penalty depends on the exact section and enforcement rules.

Why this rule exists:
Legal provisions define compliance obligations and consequences.

Advice:
Review cited sections directly for exact legal wording."""
        mode = "fallback"

    return {"answer": out, "sections": sections, "mode": mode}



In [ ]:
def format_answer(payload: Dict):
    sec = payload.get("sections", [])
    sec_text = ", ".join(sec) if sec else "Refer to applicable legal provisions"

    return f"""
LEGAL EXPLANATION:

{payload.get('answer', '').strip()}

Relevant Sections:
{sec_text}

Mode:
{payload.get('mode', 'unknown')}

Disclaimer:
This response is AI-generated legal information and not a substitute for professional legal advice.
"""



In [ ]:
print(format_answer(high_precision_answer("Penalty for not wearing helmet in India")))


In [ ]:
EVAL_SET = [
    {"query": "penalty for not wearing helmet", "must": {"129"}, "optional": {"177", "194D"}},
    {"query": "what does section 129 say", "must": {"129"}, "optional": {"177", "194D"}},
]



In [ ]:
def evaluate(eval_set):
    rows = []
    for item in eval_set:
        q = item["query"]
        out = high_precision_answer(q)
        sec = {x.upper() for x in out.get("sections", [])}

        must = {x.upper() for x in item.get("must", set())}
        optional = {x.upper() for x in item.get("optional", set())}

        rows.append({
            "query": q,
            "mode": out.get("mode"),
            "must_hit": must.issubset(sec),
            "optional_hit": bool(optional.intersection(sec)) if optional else True,
            "sections": sorted(list(sec)),
        })

    df = spark.createDataFrame(rows)
    df.show(truncate=False)
    df.groupBy("must_hit", "optional_hit").count().show()
    return df


eval_df = evaluate(EVAL_SET)



In [ ]:
ENABLE_FINE_TUNING = False

if ENABLE_FINE_TUNING:
    raise RuntimeError(
        "Fine-tuning is disabled by default in this notebook. "
        "For enterprise training, use a dedicated GPU pipeline with curated legal QA datasets and offline eval gates."
    )
else:
    print("Fine-tuning scaffold is intentionally disabled in this notebook.")

